# SkinAI — DenseNet121 분류 모델 학습

**Google Colab / 로컬 환경 모두 지원**
- Colab: Drive 마운트 → 레포 클론 → 심링크
- 로컬: 현재 디렉토리 그대로 사용

In [1]:
# ── 셀 1: 환경 감지 ────────────────────────────────────────────
import os
from pathlib import Path

try:
    import google.colab
    IS_COLAB = True
    # Colab 런타임 임시 경로 (세션 종료 시 소실)
    COLAB_ROOT = "/content/colab_skin_ai"
    PROJECT_ROOT = COLAB_ROOT
except ImportError:
    IS_COLAB = False
    # 로컬: 노트북이 위치한 레포 루트 그대로 사용
    LOCAL_ROOT = str(Path.cwd())
    PROJECT_ROOT = LOCAL_ROOT

print(f"환경       : {'Google Colab' if IS_COLAB else '로컬'}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

환경       : Google Colab
PROJECT_ROOT: /content/colab_skin_ai


In [2]:
# ── 셀 2: Drive 마운트 (Colab 전용) ────────────────────────────
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Drive 마운트 경로 (Colab 런타임 전용)
    DRIVE_ROOT = "/content/drive/MyDrive/skin_ai"
else:
    print("로컬 환경 — Drive 마운트 건너뜀")

Mounted at /content/drive


In [5]:
# ── 셀 3: 소스코드 clone (Colab 전용) ──────────────────────────
if IS_COLAB:
    if not Path(COLAB_ROOT).exists():
        !git clone https://github.com/kyoe-23/skin_ai.git {COLAB_ROOT}
    else:
        print(f"이미 존재 — 클론 건너뜀: {COLAB_ROOT}")
else:
    print("로컬 환경 — 클론 건너뜀")

Cloning into '/content/colab_skin_ai'...
remote: Enumerating objects: 328, done.
remote: Counting objects: 100% (328/328), done.
remote: Compressing objects: 100% (241/241), done.
remote: Total 328 (delta 118), reused 264 (delta 77), pack-reused 0 (from 0)
Receiving objects: 100% (328/328), 5.48 MiB | 12.76 MiB/s, done.
Resolving deltas: 100% (118/118), done.


In [6]:
# ── 셀 4: 프로젝트 루트 이동 + 데이터 경로 설정 ────────────────
os.chdir(PROJECT_ROOT)
print(f"현재 디렉토리: {os.getcwd()}")

if IS_COLAB:
    # Colab 런타임: Drive에 있는 dataset_14를 심링크로 연결
    os.makedirs("data", exist_ok=True)
    DRIVE_DATASET = f"{DRIVE_ROOT}/data/dataset_14"
    if Path(DRIVE_DATASET).exists():
        !ln -sfn {DRIVE_DATASET} data/dataset_14
        print("data/dataset_14 심링크 생성 완료")
    else:
        print(f"경고: Drive 데이터셋 없음 — {DRIVE_DATASET}")
else:
    # 로컬: data/dataset_14 존재 여부만 확인
    if not Path("data/dataset_14").exists():
        print("경고: data/dataset_14/ 없음 — AI Hub 데이터를 먼저 배치하세요")
    else:
        print("data/dataset_14/ 확인 완료")

!ls data/

현재 디렉토리: /content/colab_skin_ai
data/dataset_14 심링크 생성 완료
dataset_14  processed


In [7]:
# ── 셀 5: 패키지 설치 ───────────────────────────────────────────
!pip install -q \
    torch torchvision \
    pandas pillow tqdm \
    matplotlib python-dotenv scikit-learn

# Grad-CAM (추론 서버 사용 시)
# !pip install -q pytorch-grad-cam

In [8]:
# ── 셀 6: 학습 실행 ─────────────────────────────────────────────
# CSV의 zip_path(로컬 절대경로)를 --root_dir로 실행 환경 경로에 맞게 재매핑

# DenseNet121 (기본)
!python -m ai.training.classifier.train \
    --backbone densenet121 \
    --num_epochs 10 \
    --root_dir {PROJECT_ROOT}

# EfficientNet-B3 (비교)
# !python -m ai.training.classifier.train \
#     --backbone efficientnet_b3 \
#     --num_epochs 30 \
#     --root_dir {PROJECT_ROOT}

# 세션 만료 후 재개
# !python -m ai.training.classifier.train \
#     --backbone densenet121 \
#     --resume ai/checkpoints/aihub/best.pth \
#     --root_dir {PROJECT_ROOT}

INFO [INFO] CUDA 사용
AI Hub 08-14 분류 모델 학습
  backbone : densenet121
  device   : cuda
  epochs   : 10
  batch    : 32
  lr       : 0.001
  warmup   : 3 epochs
  scheduler: CosineAnnealingLR (T_max=7)
INFO Dataset 로드: data/processed/train.csv (4800건, direction=front)
INFO Dataset 로드: data/processed/val.csv (600건, direction=front)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()

  Train: 4800건
  Val  : 600건
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100% 30.8M/30.8M [00:0

In [ ]:
# ── 셀 7: 평가 + Threshold 최적화 ──────────────────────────────
!python -m ai.testing.evaluate \
    --checkpoint ai/checkpoints/aihub/best.pth \
    --root_dir {PROJECT_ROOT}

!python -m ai.testing.threshold_opt \
    --checkpoint ai/checkpoints/aihub/best.pth \
    --root_dir {PROJECT_ROOT}

In [ ]:
# ── 셀 8: 체크포인트 Drive 저장 (Colab 전용) ───────────────────
# 런타임 종료 전 반드시 실행 — 저장하지 않으면 학습 결과 소실
if IS_COLAB:
    import shutil
    CKPT_SRC = f"{COLAB_ROOT}/ai/checkpoints/aihub"
    CKPT_DST = f"{DRIVE_ROOT}/ai/checkpoints/aihub"
    shutil.copytree(CKPT_SRC, CKPT_DST, dirs_exist_ok=True)
    print(f"체크포인트 저장 완료: {CKPT_DST}")
else:
    print("로컬 환경 — 체크포인트 이미 로컬에 저장됨")